In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install requests pandas pyarrow -q
# import ror_enrichment

In [ ]:
#!/usr/bin/env python3
"""
ror_enrichment.py  —  Google Colab
==================================
Resolve every ROR you hold (from the re-harvested au_ror_vec) via the ROR REST
API, then BACK-FILL the author-country codes OpenAlex left blank.

WHY THIS IS THE HIGH-VALUE ONE
  OpenAlex fails to attach a country to some institutions, disproportionately in
  the Global South. Those blanks inflate the parachute rate and depress the
  local-authorship share — exactly the quantities Paper 3 measures. ROR carries a
  clean country for essentially every institution it lists, so joining on the
  RORs you already have closes the gap and lets you REPORT the residual
  missing-country rate per bloc (a required robustness number).

WHAT IT PRODUCES
  1. ror_lookup.jsonl   — one record per unique ROR: {ror, country, type, lat, lng, name}
                          (resumable cache; rerun to continue after a disconnect)
  2. adds to the table, per paper:
        au_country_vec_filled   au_country_vec with blanks back-filled from ROR
        first_author_countries_filled / last_author_countries_filled
        all_author_countries_filled
        n_authors_no_country_before / _after
  3. prints a coverage report: missing author-country rate by bloc, before vs after.

INPUT: the table that has au_ror_vec + au_country_vec + study_bloc
       (with_funding.parquet, or whichever is your latest merged parquet).

SETUP
  !pip install requests pandas pyarrow -q
"""

import json, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd, requests

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_CSV  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/authorships_reharvested.csv"
OUT_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/ror_enrich.parquet"
ROR_CACHE   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/ror_enrich/ror_lookup.jsonl"
WORKERS     = 8
MAILTO      = "cognosxx@gmail.com"   # ROR asks for a UA; be polite

MAGHREB = {"MA","DZ","TN","LY","EG","MR"}
SSA = {"AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
       "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
       "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM","ZW"}


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def short_ror(x):
    # accept 'https://ror.org/053cj9d07' or bare '053cj9d07'
    return S(x).rsplit("/", 1)[-1].strip()

def bloc_of(iso):
    return "maghreb" if iso in MAGHREB else ("ssa" if iso in SSA else "other")


# =========================================================================== #
# 1. collect every unique ROR from au_ror_vec                                  #
# =========================================================================== #
def all_rors(df):
    rors = set()
    for v in df["au_ror_vec"].dropna():
        # '|' between authors, ';' within an author
        for author in S(v).split("|"):
            for r in author.split(";"):
                r = short_ror(r)
                if r:
                    rors.add(r)
    print(f"[ror] {len(rors):,} unique RORs to resolve", file=sys.stderr)
    return rors


# =========================================================================== #
# 2. resolve via ROR REST API (resumable)                                      #
# =========================================================================== #
def load_cache():
    done = {}
    if Path(ROR_CACHE).exists():
        for line in open(ROR_CACHE):
            try:
                d = json.loads(line); done[d["ror"]] = d
            except Exception:
                pass
    print(f"[ror] {len(done):,} already cached", file=sys.stderr)
    return done

def _resolve(ror, sess):
    for attempt in range(4):
        try:
            r = sess.get(f"https://api.ror.org/organizations/{ror}", timeout=30)
            if r.status_code == 200:
                j = r.json()
                # ROR v1 schema: country.country_code; types[]; addresses[].lat/lng
                cc = (j.get("country", {}) or {}).get("country_code", "") or ""
                types = j.get("types", []) or []
                addr = (j.get("addresses", []) or [{}])[0]
                return {"ror": ror, "country": cc,
                        "type": (types[0].lower() if types else ""),
                        "lat": addr.get("lat"), "lng": addr.get("lng"),
                        "name": j.get("name", "")}
            if r.status_code == 404:
                return {"ror": ror, "country": "", "type": "", "lat": None,
                        "lng": None, "name": "", "note": "404"}
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return {"ror": ror, "country": "", "type": "", "lat": None,
                    "lng": None, "name": "", "note": str(r.status_code)}
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return {"ror": ror, "country": "", "type": "", "lat": None, "lng": None,
            "name": "", "note": "failed"}

def resolve_all(rors):
    cache = load_cache()
    todo = [r for r in rors if r not in cache]
    print(f"[ror] {len(todo):,} to fetch", file=sys.stderr)
    sess = requests.Session()
    sess.headers.update({"User-Agent": f"tick-sos-study/1.0 (mailto:{MAILTO})"})
    fh = open(ROR_CACHE, "a")
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(_resolve, r, sess) for r in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            d = fut.result()
            fh.write(json.dumps(d) + "\n"); fh.flush()
            cache[d["ror"]] = d
            if n % 500 == 0:
                print(f"[ror] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return cache


# =========================================================================== #
# 3. back-fill author-country vectors                                          #
# =========================================================================== #
def fill_row(row, ror2country):
    """Rebuild per-author country using ROR where OpenAlex gave nothing.
    au_ror_vec and au_country_vec are aligned: '|' between authors, ';' within."""
    ror_authors = S(row.get("au_ror_vec")).split("|")
    cty_authors = S(row.get("au_country_vec")).split("|")
    positions   = S(row.get("au_position_vec")).split("|")
    n = max(len(ror_authors), len(cty_authors), len(positions))

    filled, before_missing, after_missing = [], 0, 0
    for i in range(n):
        cur = cty_authors[i] if i < len(cty_authors) else ""
        cur_set = [c for c in cur.split(";") if c]
        if not cur_set:
            before_missing += 1
            rr = ror_authors[i] if i < len(ror_authors) else ""
            got = []
            for r in rr.split(";"):
                r = short_ror(r)
                cc = (ror2country.get(r, {}) or {}).get("country", "")
                if cc:
                    got.append(cc)
            cur_set = list(dict.fromkeys(got))
            if not cur_set:
                after_missing += 1
        filled.append(";".join(cur_set))

    # recompute position-based sets from the FILLED vector
    def at(kind):
        s = set()
        for i, p in enumerate(positions):
            if p == kind and i < len(filled):
                s.update(x for x in filled[i].split(";") if x)
        return s
    first_c = at("first")
    last_c = at("last")
    if len([p for p in positions if p]) == 1 and not last_c:
        last_c = set(first_c)
    all_c = [x for slot in filled for x in slot.split(";") if x]

    return {
        "au_country_vec_filled": "|".join(filled),
        "first_author_countries_filled": ";".join(sorted(first_c)),
        "last_author_countries_filled": ";".join(sorted(last_c)),
        "all_author_countries_filled": "|".join(all_c),
        "n_authors_no_country_before": before_missing,
        "n_authors_no_country_after": after_missing,
    }


def dominant_bloc(sb):
    parts = {x for x in S(sb).split(";") if x}
    for b in ("ssa", "maghreb"):
        if b in parts: return b
    return "other"

def coverage_report(df):
    """Missing author-country rate by bloc, before vs after — the robustness number."""
    if "study_bloc" not in out.columns:
        print("[validate] study_bloc absent — run on a parquet with study-location.",
              file=sys.stderr)
        return

    df["study_bloc"] = df["study_bloc"].map(dominant_bloc)

    def rate(col):
        rows = []
        for blc in ("ssa", "maghreb", "other"):
            m = df["study_bloc"].fillna("").str.contains(blc)
            sub = df[m]
            if len(sub):
                rows.append((blc, len(sub), round(sub[col].mean(), 3)))
        return rows
    print("\n[coverage] mean authors-with-no-country per paper, by bloc:")
    print(f"{'bloc':<9}{'n_papers':>10}{'before':>9}{'after':>9}")
    before = {b: (n, r) for b, n, r in rate("n_authors_no_country_before")}
    after  = {b: (n, r) for b, n, r in rate("n_authors_no_country_after")}
    for b in ("ssa", "maghreb", "other"):
        if b in before:
            print(f"{b:<9}{before[b][0]:>10}{before[b][1]:>9}{after[b][1]:>9}")


# =========================================================================== #
if __name__ == "__main__":
    # Ensure the directory for ROR_CACHE exists
    Path(ROR_CACHE).parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(IN_CSV)
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    cache = resolve_all(all_rors(df))
    filled = df.apply(lambda r: fill_row(r, cache), axis=1, result_type="expand")
    out = pd.concat([df, filled], axis=1)
    # Ensure the directory for OUT_PARQUET exists
    Path(OUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)
    coverage_report(out)


In [ ]:
out